# NOVA — Existing Model Artifact Audit (Read-only)

**Purpose:** inventory and inspect existing NOVA model artifacts in Google Drive before any retraining or deployment.

## Safety contract
- This notebook is **AUDIT ONLY**. It does not train, fine-tune, export, overwrite, delete, rename, or upload files.
- Google Drive is mounted with `readonly=True`.
- No broker, MT5, Controller, execution, or trading API is called.
- No model inference is run. ONNX is inspected structurally only; PyTorch checkpoints are loaded with `weights_only=True` where supported to inspect tensor names and shapes.
- Do not change any safety flags to `True`. Training, Controller, Execution, and Live Trading remain OFF.
- A file's presence or tensor structure does **not** prove model quality, provenance, OOS performance, or production readiness.

Run cells in order. If Drive mount fails, stop and resolve access; do not substitute mock files.

In [ ]:
# Immutable safety declarations for this audit
MODE = "READ_ONLY_AUDIT"
TRAINING_ENABLED = False
MODEL_EXPORT_ENABLED = False
ARTIFACT_WRITE_ENABLED = False
CONTROLLER_ENABLED = False
EXECUTION_ENABLED = False
LIVE_TRADING_ENABLED = False

assert MODE == "READ_ONLY_AUDIT"
assert not any([TRAINING_ENABLED, MODEL_EXPORT_ENABLED, ARTIFACT_WRITE_ENABLED,
                CONTROLLER_ENABLED, EXECUTION_ENABLED, LIVE_TRADING_ENABLED])
print({"mode": MODE, "training": TRAINING_ENABLED, "artifact_write": ARTIFACT_WRITE_ENABLED,
       "controller": CONTROLLER_ENABLED, "execution": EXECUTION_ENABLED,
       "live_trading": LIVE_TRADING_ENABLED})

## 1. Mount Google Drive read-only
Authenticate with your Google account in the Colab prompt. Do not share credentials or OTP codes in chat.

In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path('/content/drive')
drive.mount(str(DRIVE_MOUNT), readonly=True)
NOVA_ROOT = DRIVE_MOUNT / 'MyDrive' / 'NOVA'
print('Drive mounted:', DRIVE_MOUNT.exists())
print('NOVA root exists:', NOVA_ROOT.exists())
if not NOVA_ROOT.exists():
    raise FileNotFoundError(f'NOVA directory not found: {NOVA_ROOT}. Stop; do not create it in this audit notebook.')

## 2. Inventory existing model and metadata files
The scan is limited to the NOVA Drive folder. It reports paths, sizes, timestamps, and SHA256; it does not modify files.

In [ ]:
import hashlib, json, os
from datetime import datetime, timezone

MODEL_EXTENSIONS = {'.onnx', '.pt', '.pth', '.ckpt', '.safetensors', '.pb', '.h5', '.keras'}
METADATA_NAMES = {'metadata.json', 'model_metadata.json', 'manifest.json', 'config.json',
                  'feature_schema.json', 'metrics.json', 'training_config.json'}
EXCLUDED_DIRS = {'.Trash-1000', '.ipynb_checkpoints', '__pycache__'}

def sha256_file(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

inventory = []
for current, dirs, files in os.walk(NOVA_ROOT):
    dirs[:] = [d for d in dirs if d not in EXCLUDED_DIRS]
    for filename in files:
        path = Path(current) / filename
        if path.suffix.lower() in MODEL_EXTENSIONS or filename.lower() in METADATA_NAMES:
            try:
                stat = path.stat()
                inventory.append({
                    'relative_path': str(path.relative_to(NOVA_ROOT)),
                    'extension': path.suffix.lower(),
                    'size_bytes': stat.st_size,
                    'modified_utc': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
                    'sha256': sha256_file(path),
                })
            except Exception as exc:
                inventory.append({'relative_path': str(path.relative_to(NOVA_ROOT)), 'error': repr(exc)})

inventory = sorted(inventory, key=lambda item: item['relative_path'].lower())
print('Artifact/metadata files found:', len(inventory))
for item in inventory:
    print(json.dumps(item, ensure_ascii=False))
if not inventory:
    print('NO MODEL OR METADATA ARTIFACTS FOUND under NOVA_ROOT.')

## 3. Inspect JSON sidecar metadata
Only parse known JSON metadata files. Invalid JSON is reported, never repaired or rewritten.

In [ ]:
metadata_report = []
for item in inventory:
    rel = item.get('relative_path', '')
    if Path(rel).name.lower() in METADATA_NAMES and 'error' not in item:
        path = NOVA_ROOT / rel
        try:
            with path.open('r', encoding='utf-8') as stream:
                payload = json.load(stream)
            metadata_report.append({'path': rel, 'valid_json': True,
                                    'top_level_keys': list(payload.keys()) if isinstance(payload, dict) else type(payload).__name__,
                                    'content': payload})
        except Exception as exc:
            metadata_report.append({'path': rel, 'valid_json': False, 'error': repr(exc)})
for item in metadata_report:
    print(json.dumps(item, ensure_ascii=False, indent=2, default=str))
if not metadata_report:
    print('No recognized JSON sidecar metadata files found.')

## 4. ONNX structural inspection (no inference)
If `onnx` is installed, inspect graph inputs/outputs, opset, and metadata. This does not create an inference session or run predictions.

In [ ]:
onnx_report = []
onnx_paths = [NOVA_ROOT / x['relative_path'] for x in inventory
              if x.get('extension') == '.onnx' and 'error' not in x]
try:
    import onnx
except ImportError:
    onnx = None
    print('onnx package unavailable; structural ONNX inspection skipped. No install performed.')

if onnx is not None:
    for path in onnx_paths:
        try:
            model = onnx.load(str(path), load_external_data=False)
            onnx.checker.check_model(model)
            def value_info(v):
                tensor = v.type.tensor_type
                dims = [d.dim_value if d.HasField('dim_value') else d.dim_param for d in tensor.shape.dim]
                return {'name': v.name, 'elem_type': tensor.elem_type, 'shape': dims}
            report = {'path': str(path.relative_to(NOVA_ROOT)),
                      'ir_version': model.ir_version,
                      'opsets': [{'domain': x.domain, 'version': x.version} for x in model.opset_import],
                      'inputs': [value_info(v) for v in model.graph.input],
                      'outputs': [value_info(v) for v in model.graph.output],
                      'metadata': {x.key: x.value for x in model.metadata_props},
                      'checker': 'PASS'}
        except Exception as exc:
            report = {'path': str(path.relative_to(NOVA_ROOT)), 'checker': 'FAIL', 'error': repr(exc)}
        onnx_report.append(report)
        print(json.dumps(report, ensure_ascii=False, indent=2, default=str))
if not onnx_paths:
    print('No ONNX files found.')

## 5. PyTorch checkpoint structure inspection (no inference)
Loads tensor state dictionaries only. `weights_only=True` is required; if the installed PyTorch version does not support it, the checkpoint is skipped rather than loaded unsafely.

In [ ]:
torch_report = []
torch_paths = [NOVA_ROOT / x['relative_path'] for x in inventory
               if x.get('extension') in {'.pt', '.pth', '.ckpt'} and 'error' not in x]
try:
    import torch
except ImportError:
    torch = None
    print('PyTorch unavailable; checkpoint inspection skipped. No install performed.')

if torch is not None:
    for path in torch_paths:
        try:
            payload = torch.load(str(path), map_location='cpu', weights_only=True)
            state = payload.get('state_dict', payload) if isinstance(payload, dict) else payload
            if not isinstance(state, dict):
                raise TypeError('Checkpoint is not a state-dict-like mapping')
            tensors = {}
            for key, value in state.items():
                if torch.is_tensor(value):
                    tensors[str(key)] = {'shape': list(value.shape), 'dtype': str(value.dtype)}
            report = {'path': str(path.relative_to(NOVA_ROOT)),
                      'safe_load': 'PASS', 'tensor_count': len(tensors),
                      'tensors': tensors}
        except TypeError as exc:
            report = {'path': str(path.relative_to(NOVA_ROOT)), 'safe_load': 'SKIPPED',
                      'reason': 'weights_only=True unsupported or incompatible checkpoint', 'detail': repr(exc)}
        except Exception as exc:
            report = {'path': str(path.relative_to(NOVA_ROOT)), 'safe_load': 'FAIL_OR_SKIPPED', 'error': repr(exc)}
        torch_report.append(report)
        print(json.dumps(report, ensure_ascii=False, indent=2, default=str))
if not torch_paths:
    print('No PyTorch checkpoint files found.')

## 6. Audit summary and stop conditions
This summary is descriptive only. It does not approve training or deployment.

In [ ]:
onnx_count = len(onnx_paths)
torch_count = len(torch_paths)
metadata_count = len(metadata_report)
print('=== NOVA READ-ONLY MODEL AUDIT SUMMARY ===')
print('Inventory entries:', len(inventory))
print('ONNX files:', onnx_count)
print('PyTorch checkpoint files:', torch_count)
print('Metadata files inspected:', metadata_count)
print('ONNX structural checks:', sum(x.get('checker') == 'PASS' for x in onnx_report), 'PASS /',
      sum(x.get('checker') == 'FAIL' for x in onnx_report), 'FAIL')
print('Training: NOT RUN')
print('Inference: NOT RUN')
print('Artifact writes: NONE')
print('Controller: OFF | Execution: OFF | Live Trading: OFF')
print('\nIMPORTANT: Presence + hash + structural compatibility are not proof of provenance, predictive quality, OOS performance, or production readiness.')
print('Next gate: manually reconcile model hashes, metadata, feature schema, dataset lineage, and evaluation reports before any separately approved experiment.')

## Audit completion checklist
- [ ] Confirm Drive mounted read-only.
- [ ] Review full inventory and SHA256 values.
- [ ] Match each model to metadata, version, feature schema, and training provenance.
- [ ] Confirm ONNX input/output dimensions match the approved NOVA feature contract.
- [ ] Locate evaluation reports and verify a genuine untouched OOS split.
- [ ] Keep Training, Controller, Execution, and Live Trading OFF until separately authorized.

**Do not add training cells to this notebook.** Any future training experiment must use a different notebook, a separately approved dataset, a frozen configuration, and isolated output paths.